#  Task 5: Model Persistence and Advanced Techniques

Name : Luqman Khan

SID : S202266054

Section : V1

Course Code: CS4152

Semester: Fall 2026

Sumitted to: Dr. Jameel Ahmad

Course Title: Deep Learning and Neural Networks

In [54]:
# Setup
import os
import random
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import r2_score


# **Step 1: Create a folder for saved models**

In [55]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

os.makedirs('saved_models', exist_ok=True)

Explanation:
- os.makedirs() creates a folder named saved_models.
- exist_ok=True prevents an error if the folder already exists.
- We will store our trained models inside this folder.

In [56]:
# Task 1 Data Preparation
df = pd.read_csv('/content/taxi-fares.csv', parse_dates=['pickup_datetime'])
df = df[df['passenger_count'] == 1]

df['day_of_week'] = df['pickup_datetime'].dt.weekday
df['pickup_time'] = df['pickup_datetime'].dt.hour

dx = (df['dropoff_longitude'] - df['pickup_longitude']) * 54.6
dy = (df['dropoff_latitude'] - df['pickup_latitude']) * 69.0
df['distance'] = np.sqrt(dx ** 2 + dy ** 2)

df = df[(df['distance'] > 1.0) & (df['distance'] < 10.0)]
df = df[(df['fare_amount'] > 0.0) & (df['fare_amount'] < 50.0)]

x = df[['day_of_week', 'pickup_time', 'distance']].values
y = df['fare_amount'].values
print('Data shape:', x.shape, y.shape)

Data shape: (23298, 3) (23298,)


In [57]:
# Task 1 Model
def build_taxi_model(width=512):
    model = Sequential([
        Input(shape=(3,)),
        Dense(width, activation='relu'),
        Dense(width, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mae', metrics=['mae'])
    return model

# **Step 2: Create the callbacks**

In [58]:
early_stop = EarlyStopping(
    monitor='val_mae',
    mode='min',
    patience=8,
    restore_best_weights=True
)

**Explanation of each parameter:**
if validation MAE improves until epoch 20 and then does not improve for eight epochs, training can stop instead of continuing all 100 epochs.

In [59]:
checkpoint = ModelCheckpoint(
    'saved_models/best_taxi_checkpoint.h5',
    monitor='val_mae',
    mode='min',
    save_best_only=True
)

In [60]:
best_model = build_taxi_model(512)

hist = best_model.fit(
    x,
    y,
    validation_split=0.2,
    epochs=100,
    batch_size=100,
    callbacks=[early_stop, checkpoint],
    verbose=1
)

Epoch 1/100
187/187 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 4.0745 - mae: 4.0745

187/187 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - loss: 2.8967 - mae: 2.8967 - val_loss: 2.3768 - val_mae: 2.3768
Epoch 2/100
185/187 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 2.4186 - mae: 2.4186

187/187 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 2.3911 - mae: 2.3911 - val_loss: 2.3637 - val_mae: 2.3637
Epoch 3/100
181/187 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 2.4074 - mae: 2.4074

187/187 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 2.3868 - mae: 2.3868 - val_loss: 2.3600 - val_mae: 2.3600
Epoch 4/100
181/187 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 2.3926 - mae: 2.3926

187/187 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 2.3793 - mae: 2.3793 - val_loss: 2.3587 - val_mae: 2.3587
Epoch 5/100
185/187 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 2.3917 - mae: 2.3917

...

187/187 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - loss: 2.2903 - mae: 2.2903 - val_loss: 2.2595 - val_mae: 2.2595
Epoch 69/100
187/187 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 2.2920 - mae: 2.2920 - val_loss: 2.2606 - val_mae: 2.2606
Epoch 70/100
187/187 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 2.3026 - mae: 2.3026

187/187 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 2.2883 - mae: 2.2883 - val_loss: 2.2568 - val_mae: 2.2568
Epoch 71/100
187/187 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 2.2927 - mae: 2.2927 - val_loss: 2.2628 - val_mae: 2.2628
Epoch 72/100
187/187 ━━━━━━━━━━━━━━━━━━━━ 4s 19ms/step - loss: 2.2910 - mae: 2.2910 - val_loss: 2.2584 - val_mae: 2.2584
Epoch 73/100
187/187 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - loss: 2.2891 - mae: 2.2891 - val_loss: 2.2586 - val_mae: 2.2586
Epoch 74/100
184/187 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 2.3035 - mae: 2.3035

187/187 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 2.2893 - mae: 2.2893 - val_loss: 2.2559 - val_mae: 2.2559
Epoch 75/100
181/187 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 2.3036 - mae: 2.3036

187/187 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 2.2891 - mae: 2.2891 - val_loss: 2.2555 - val_mae: 2.2555
Epoch 76/100
187/187 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 2.2885 - mae: 2.2885 - val_loss: 2.2558 - val_mae: 2.2558
Epoch 77/100
187/187 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - loss: 2.2895 - mae: 2.2895 - val_loss: 2.2613 - val_mae: 2.2613
Epoch 78/100
186/187 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 2.3027 - mae: 2.3027

187/187 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - loss: 2.2887 - mae: 2.2887 - val_loss: 2.2539 - val_mae: 2.2539
Epoch 79/100
187/187 ━━━━━━━━━━━━━━━━━━━━ 5s 26ms/step - loss: 2.2877 - mae: 2.2877 - val_loss: 2.2630 - val_mae: 2.2630
Epoch 80/100
187/187 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 2.2903 - mae: 2.2903 - val_loss: 2.2632 - val_mae: 2.2632
Epoch 81/100
187/187 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - loss: 2.2858 - mae: 2.2858 - val_loss: 2.2566 - val_mae: 2.2566
Epoch 82/100
187/187 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 2.2906 - mae: 2.2906 - val_loss: 2.2634 - val_mae: 2.2634
Epoch 83/100
187/187 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - loss: 2.2892 - mae: 2.2892 - val_loss: 2.2606 - val_mae: 2.2606
Epoch 84/100
187/187 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 2.2881 - mae: 2.2881 - val_loss: 2.2617 - val_mae: 2.2617
Epoch 85/100
187/187 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - loss: 2.2860 - mae: 2.2860 - val_loss: 2.2737 - val_mae: 2.2737
Epoch 86/100
187/187 ━━━━━━━━━━━━━━━━━━━━ 3s 10m

In [61]:
pd.DataFrame(hist.history).to_csv('saved_models/task5_history.csv', index=False)

epochs_run = len(hist.history['val_mae'])
best_epoch = int(np.argmin(hist.history['val_mae'])) + 1
print('Maximum epochs allowed      :', 100)
print('Epochs actually run         :', epochs_run)
print('Epochs saved by early stop  :', 100 - epochs_run)
print('Best epoch                  :', best_epoch)
print('Best validation MAE         : %.4f' % min(hist.history['val_mae']))
print('Last-epoch validation MAE   : %.4f' % hist.history['val_mae'][-1])

Maximum epochs allowed      : 100
Epochs actually run         : 86
Epochs saved by early stop  : 14
Best epoch                  : 78
Best validation MAE         : 2.2539
Last-epoch validation MAE   : 2.2648


# step 3: Model Saving (H5 and SavedModel formats)

In [62]:
best_model.save('saved_models/taxi_model.h5')
best_model.export('saved_models/taxi_model_savedmodel')

for root, folders, files in os.walk('saved_models'):
    for name in files:
        path = os.path.join(root, name)
        print('%-70s %10.1f KB' % (path, os.path.getsize(path) / 1024))

Saved artifact at 'saved_models/taxi_model_savedmodel'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 3), dtype=tf.float32, name='keras_tensor_30')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  132866929060432: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132866929061392: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132866929061200: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132866929061776: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132866929061584: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132866929062160: TensorSpec(shape=(), dtype=tf.resource, name=None)
saved_models/taxi_model.h5                                                 3140.2 KB
saved_models/best_taxi_checkpoint.h5                                       3140.2 KB
saved_models/task5_history.csv                                                6.3 KB
saved_models/taxi_model_savedm

# step 4: Model Loading and Inference

In [67]:
h5_model = load_model('saved_models/taxi_model.h5', compile=False)
h5_model.compile(optimizer='adam', loss='mae', metrics=['mae'])

savedmodel = tf.saved_model.load('saved_models/taxi_model_savedmodel')

checkpoint_model = None
for name in ['best_taxi_checkpoint.h5', 'best_taxi_checkpoint.keras']:
    path = os.path.join('saved_models', name)
    if os.path.exists(path):
        checkpoint_model = load_model(path, compile=False)
        checkpoint_model.compile(optimizer='adam', loss='mae', metrics=['mae'])
        print('Loaded checkpoint:', path)
        break


def savedmodel_predict(loaded, data):
    data = tf.constant(data, dtype=tf.float32)
    if hasattr(loaded, 'serve'):
        out = loaded.serve(data)
    else:
        out = list(loaded.signatures['serving_default'](data).values())[0]
    return np.array(out)


new_data = np.array([
    [4, 17, 2.0],
    [5, 17, 2.0],
    [0, 9, 5.0],
    [6, 23, 8.0]
])

pred_original = best_model.predict(new_data, verbose=0).flatten()
pred_h5 = h5_model.predict(new_data, verbose=0).flatten()
pred_savedmodel = savedmodel_predict(savedmodel, new_data).flatten()


Loaded checkpoint: saved_models/best_taxi_checkpoint.h5


In [68]:
print('Max difference H5         : %.8f' % np.max(np.abs(pred_original - pred_h5)))
print('Max difference SavedModel : %.8f' % np.max(np.abs(pred_original - pred_savedmodel)))
print('Match (H5)         :', np.allclose(pred_original, pred_h5, atol=1e-4))
print('Match (SavedModel) :', np.allclose(pred_original, pred_savedmodel, atol=1e-4))

if checkpoint_model is not None:
    pred_checkpoint = checkpoint_model.predict(new_data, verbose=0).flatten()
    print('Max difference Checkpoint : %.8f' % np.max(np.abs(pred_original - pred_checkpoint)))
    print('Match (Checkpoint) :', np.allclose(pred_original, pred_checkpoint, atol=1e-4))

results = pd.DataFrame({
    'day': new_data[:, 0],
    'hour': new_data[:, 1],
    'distance': new_data[:, 2],
    'original': pred_original,
    'loaded_h5': pred_h5,
    'loaded_savedmodel': pred_savedmodel
})
print(results.round(4).to_string(index=False))

print('R2 original model  :', r2_score(y, best_model.predict(x, verbose=0).flatten()))
print('R2 loaded H5 model :', r2_score(y, h5_model.predict(x, verbose=0).flatten()))

Max difference H5         : 0.00000000
Max difference SavedModel : 0.00000000
Match (H5)         : True
Match (SavedModel) : True
Max difference Checkpoint : 0.00000000
Match (Checkpoint) : True
 day  hour  distance  original  loaded_h5  loaded_savedmodel
 4.0  17.0       2.0   10.5093    10.5093            10.5093
 5.0  17.0       2.0   10.3230    10.3230            10.3230
 0.0   9.0       5.0   24.0795    24.0795            24.0795
 6.0  23.0       8.0   29.7966    29.7966            29.7966
R2 original model  : 0.7490835155494737
R2 loaded H5 model : 0.7490835155494737


# Explaination of code

The code takes the best Task 1 taxi-fare model (3 inputs, two hidden layers of 512 ReLU neurons, 1 output), trains it with callbacks, saves it in several formats, loads it back, and checks that nothing changed.

1. Setup and data preparation
It sets a random seed (42) for reproducible results and creates a saved_models folder. It then loads the taxi data and keeps single-passenger trips, with distance between 1 and 10 miles and fare between $0 and $50. The inputs are day of week, pickup hour and distance, and the target is the fare.

2. Part (c): callbacks

EarlyStopping watches validation MAE. It stops training if there is no improvement for 8 epochs and restores the best weights.
ModelCheckpoint saves the model to a file only when validation MAE improves, so the best version is always kept.
The model trains for up to 100 epochs with batch size 100 and a validation split of 0.2. The code prints how many epochs ran, how many were saved, the best epoch and the best validation MAE.

3. Part (a): saving the model

model.save('taxi_model.h5') creates a single legacy Keras file.
model.export('taxi_model_savedmodel') creates a TensorFlow SavedModel folder, which other tools and languages can also use.
The code lists the saved files with their sizes.

4. Part (b): loading and verification

It loads the H5 model, the SavedModel and the checkpoint file.
All of them predict the fare for four new trips, including the Friday and Saturday 5 PM cases.
It compares each loaded model’s predictions with the original model’s, using the maximum difference and np.allclose.
It also compares the R2 score of the original and loaded models.

Your results

All three maximum differences were 0.00000000, and all three match checks were True.
R2 was 0.749 for both the original and the loaded model.
So saving and loading keeps the model’s weights and biases exactly, and the loaded models can be used for predictions without retraining.